# 02 Causal Mask

这个 notebook 用一个小例子观察 GPT 这类 decoder-only 模型为什么不能看见未来 token。

目标：理解 causal mask 如何把未来位置的 attention score 变成 `-inf`，从而让 softmax 后的概率变成 0。

In [ ]:
import math

import matplotlib.pyplot as plt
import torch

torch.manual_seed(3)
torch.set_printoptions(precision=3, sci_mode=False)

## 1. 构造 query/key 并计算原始分数

`scores[i, j]` 表示第 `i` 个位置对第 `j` 个位置的注意力分数。

In [ ]:
seq, dim = 5, 4

q = torch.randn(seq, dim)
k = torch.randn(seq, dim)

scores = q @ k.T / math.sqrt(dim)

print("scores.shape:", tuple(scores.shape))
scores

## 2. 构造 causal mask

下三角为 1，表示当前位置可以看自己和过去；上三角为 0，表示不能看未来。

In [ ]:
causal_mask = torch.tril(torch.ones(seq, seq, dtype=torch.bool))

causal_mask.int()

## 3. 应用 mask 并 softmax

被 mask 掉的位置先变成 `-inf`，softmax 后对应概率就是 0。

In [ ]:
masked_scores = scores.masked_fill(~causal_mask, float("-inf"))
weights = torch.softmax(masked_scores, dim=-1)

print("masked scores:")
print(masked_scores)
print("\nattention weights:")
print(weights)

## 4. 可视化 mask 前后的变化

第三张图里的上三角应该全部接近 0，这就是 GPT 训练和生成时的“不能看未来”。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

items = [
    (scores, "raw scores"),
    (causal_mask.int(), "visible positions"),
    (weights, "masked attention weights"),
]

for ax, (matrix, title) in zip(axes, items):
    im = ax.imshow(matrix.detach(), cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("key position")
    ax.set_ylabel("query position")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 5. 对比：没有 mask 会发生什么

如果不加 causal mask，每一行都可以把概率分配给未来位置。

In [ ]:
unmasked_weights = torch.softmax(scores, dim=-1)
future_positions = torch.triu(torch.ones(seq, seq, dtype=torch.bool), diagonal=1)

future_mass_without_mask = unmasked_weights.masked_select(future_positions).sum()
future_mass_with_mask = weights.masked_select(future_positions).sum()

print("future probability mass without mask:", float(future_mass_without_mask))
print("future probability mass with mask:   ", float(future_mass_with_mask))

## 练习

- 把 `seq` 改成 8，观察 mask 和 attention weights 的形状。
- 把 `dim` 改大，观察 raw scores 的数值变化。
- 思考：为什么训练时也要加 causal mask，而不只是生成时加？